In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact_manual

cwd = Path.cwd()
if (cwd / 'column_scaled_wishart_levy.py').exists():
    sys.path.insert(0, str(cwd))
elif (cwd / 'random_dnn' / 'column_scaled_wishart_levy.py').exists():
    sys.path.insert(0, str(cwd / 'random_dnn'))

import column_scaled_wishart_levy as cwl


In [ ]:
DEFAULT_PARAMS = dict(
    alpha=1.5,
    gamma=1.0,
    profile='constant',
    constant_value=1.0,
    left_value=0.8,
    right_value=1.6,
    split=0.4,
    exponent=0.5,
    cutoff=0.05,
    entry_scale=1.0,
    normalization='stable',
    n_rows=64,
    num_matrices=8,
    bins=81,
    s_max=6.0,
    num_points=121,
    imag_eps=1e-2,
    quadrature_order=64,
    profile_order=96,
    tail_start=2.0,
)

DEFAULT_PARAMS


In [ ]:
def CUSTOM_PROFILE(u):
    """Edit this function and then choose profile='custom_function' below."""
    u = np.asarray(u, dtype=float)
    return 1.0 + 0.35 * np.cos(2.0 * np.pi * u) ** 2

CUSTOM_PROFILE(np.linspace(0.0, 1.0, 5))


In [ ]:
def plot_column_scaled_comparison(
    alpha,
    gamma,
    profile,
    constant_value,
    left_value,
    right_value,
    split,
    exponent,
    cutoff,
    entry_scale,
    normalization,
    n_rows,
    num_matrices,
    bins,
    s_max,
    num_points,
    imag_eps,
    quadrature_order,
    profile_order,
    tail_start,
    seed,
    show_empirical=True,
    show_theory=True,
    show_tail=True,
    show_constant_check=True,
):
    params = dict(
        alpha=float(alpha),
        gamma=float(gamma),
        profile=str(profile),
        constant_value=float(constant_value),
        left_value=float(left_value),
        right_value=float(right_value),
        split=float(split),
        exponent=float(exponent),
        cutoff=float(cutoff),
        entry_scale=float(entry_scale),
        normalization=str(normalization),
        n_rows=int(n_rows),
        num_matrices=int(num_matrices),
        bins=int(bins),
        s_max=float(s_max),
        num_points=int(num_points),
        imag_eps=float(imag_eps),
        quadrature_order=int(quadrature_order),
        profile_order=int(profile_order),
        tail_start=float(tail_start),
        seed=int(seed),
    )

    profile_spec = CUSTOM_PROFILE if params['profile'] == 'custom_function' else params['profile']

    profile_kwargs = dict(
        constant_value=params['constant_value'],
        left_value=params['left_value'],
        right_value=params['right_value'],
        split=params['split'],
        exponent=params['exponent'],
        cutoff=params['cutoff'],
    )

    theory = (
        cwl.theoretical_column_scaled_singular_value_curve(
            params['alpha'],
            params['gamma'],
            profile=profile_spec,
            s_max=params['s_max'],
            num_points=params['num_points'],
            entry_scale=params['entry_scale'],
            normalization=params['normalization'],
            imag_eps=params['imag_eps'],
            quadrature_order=params['quadrature_order'],
            profile_order=params['profile_order'],
            **profile_kwargs,
        )
        if show_theory else None
    )

    empirical = (
        cwl.empirical_column_scaled_singular_value_spectrum(
            params['alpha'],
            params['gamma'],
            profile=profile_spec,
            n_rows=params['n_rows'],
            num_matrices=params['num_matrices'],
            entry_scale=params['entry_scale'],
            normalization=params['normalization'],
            bins=params['bins'],
            seed=params['seed'],
            singular_range=(0.0, params['s_max']),
            **profile_kwargs,
        )
        if show_empirical else None
    )

    profile_grid = np.linspace(0.0, 1.0, 400)
    profile_values, profile_name = cwl.evaluate_column_profile(profile_grid, profile_spec, **profile_kwargs)

    tail_grid = np.linspace(max(params['tail_start'], 1e-3), params['s_max'], 400)
    tail_density = cwl.asymptotic_singular_density(
        tail_grid,
        params['alpha'],
        params['gamma'],
        profile_spec,
        entry_scale=params['entry_scale'],
        normalization=params['normalization'],
        profile_order=params['profile_order'],
        **profile_kwargs,
    )

    tail_pref = cwl.singular_tail_prefactor(
        params['alpha'],
        params['gamma'],
        profile_spec,
        entry_scale=params['entry_scale'],
        normalization=params['normalization'],
        profile_order=params['profile_order'],
        **profile_kwargs,
    )

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    axes[0].plot(profile_grid, profile_values, color='tab:purple', lw=2)
    axes[0].set_xlabel(r'$u$')
    axes[0].set_ylabel(r'$c(u)$')
    axes[0].set_title(profile_name)

    if show_empirical and empirical is not None:
        widths = np.diff(empirical.sv_bin_edges)
        axes[1].bar(
            empirical.sv_bin_centers,
            empirical.sv_density,
            width=widths,
            color='steelblue',
            alpha=0.35,
            label=f'empirical SVD (N={empirical.n_rows}, M={empirical.n_cols}, n={empirical.num_matrices})',
        )
    if show_theory and theory is not None:
        axes[1].plot(
            theory.singular_values,
            theory.singular_density,
            color='black',
            lw=2,
            label='column-scaled profile theory',
        )
    if show_tail:
        axes[1].plot(
            tail_grid,
            tail_density,
            color='crimson',
            ls=':',
            lw=2,
            label=rf'tail $\sim {tail_pref:.4f}\,s^{{-(1+{params["alpha"]:.2f})}}$',
        )
    axes[1].set_xlabel(r'$s$')
    axes[1].set_ylabel(r'$f_{\rm sv}(s)$')
    axes[1].set_xlim(0.0, params['s_max'])
    axes[1].set_title(rf'$\alpha={params["alpha"]}$, $\gamma={params["gamma"]}$ ({params["normalization"]})')
    axes[1].legend()

    if show_empirical and empirical is not None:
        mask = (empirical.sv_bin_centers >= params['tail_start']) & (empirical.sv_density > 0.0)
        axes[2].loglog(
            empirical.sv_bin_centers[mask],
            empirical.sv_density[mask],
            'o',
            color='steelblue',
            ms=4,
            label='empirical density',
        )
    if show_theory and theory is not None:
        mask = (theory.singular_values >= params['tail_start']) & (theory.singular_density > 0.0)
        axes[2].loglog(
            theory.singular_values[mask],
            theory.singular_density[mask],
            color='black',
            lw=2,
            label='profile theory',
        )
    if show_tail:
        axes[2].loglog(tail_grid, tail_density, color='crimson', ls=':', lw=2, label='tail asymptotic')
    axes[2].set_xlabel(r'$s$')
    axes[2].set_ylabel(r'$f_{\rm sv}(s)$')
    axes[2].set_title('Log-log tail check')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    alpha_moment = cwl.column_profile_alpha_moment(
        params['alpha'], profile_spec, profile_order=params['profile_order'], **profile_kwargs,
    )
    survival_pref = cwl.singular_survival_tail_prefactor(
        params['alpha'], params['gamma'], profile_spec, entry_scale=params['entry_scale'], normalization=params['normalization'], profile_order=params['profile_order'], **profile_kwargs,
    )
    print(f'profile alpha-moment ∫ c(u)^alpha du = {alpha_moment:.6f}')
    print(f'singular-value tail: f_sv(s) ~ {tail_pref:.6f} * s^(-(1 + {params["alpha"]:.3f}))')
    print(f'survival tail: P(S > s) ~ {survival_pref:.6f} * s^(-{params["alpha"]:.3f})')
    q = 0.99
    q_asym = cwl.asymptotic_singular_quantile(
        q, params['alpha'], params['gamma'], profile_spec, entry_scale=params['entry_scale'], normalization=params['normalization'], profile_order=params['profile_order'], **profile_kwargs,
    )
    print(f'asymptotic upper singular-value quantile Q({q:.2f}) ~ {float(q_asym):.6f}')
    if show_theory and theory is not None:
        print(f'atom at zero in squared-singular-value law: {theory.atom_at_zero:.6f}')
        print(f'theory profile label: {theory.profile_name}')
    if show_empirical and empirical is not None:
        print(f'actual matrix shape: N={empirical.n_rows}, M={empirical.n_cols}, gamma={empirical.gamma:.6f}')
    if params['profile'] == 'constant' and show_constant_check:
        cmp = cwl.compare_constant_profile_to_wishart(
            params['alpha'],
            params['gamma'],
            s_max=params['s_max'],
            num_points=params['num_points'],
            entry_scale=params['entry_scale'],
            normalization=params['normalization'],
            imag_eps=params['imag_eps'],
            quadrature_order=params['quadrature_order'],
            profile_order=params['profile_order'],
        )
        print('constant-profile consistency vs wishart_levy.py:', cmp)


In [ ]:
# plot_column_scaled_comparison(**DEFAULT_PARAMS, seed=0)

@interact_manual(
    alpha=widgets.FloatText(value=DEFAULT_PARAMS['alpha']),
    gamma=widgets.FloatText(value=DEFAULT_PARAMS['gamma']),
    profile=widgets.Dropdown(options=['constant', 'two_level', 'power_law', 'custom_function'], value=DEFAULT_PARAMS['profile']),
    constant_value=widgets.FloatText(value=DEFAULT_PARAMS['constant_value']),
    left_value=widgets.FloatText(value=DEFAULT_PARAMS['left_value']),
    right_value=widgets.FloatText(value=DEFAULT_PARAMS['right_value']),
    split=widgets.FloatText(value=DEFAULT_PARAMS['split']),
    exponent=widgets.FloatText(value=DEFAULT_PARAMS['exponent']),
    cutoff=widgets.FloatText(value=DEFAULT_PARAMS['cutoff']),
    entry_scale=widgets.FloatText(value=DEFAULT_PARAMS['entry_scale']),
    normalization=widgets.Dropdown(options=['stable', 'belinschi'], value=DEFAULT_PARAMS['normalization']),
    n_rows=widgets.IntText(value=DEFAULT_PARAMS['n_rows']),
    num_matrices=widgets.IntText(value=DEFAULT_PARAMS['num_matrices']),
    bins=widgets.IntText(value=DEFAULT_PARAMS['bins']),
    s_max=widgets.FloatText(value=DEFAULT_PARAMS['s_max']),
    num_points=widgets.IntText(value=DEFAULT_PARAMS['num_points']),
    imag_eps=widgets.FloatText(value=DEFAULT_PARAMS['imag_eps']),
    quadrature_order=widgets.IntText(value=DEFAULT_PARAMS['quadrature_order']),
    profile_order=widgets.IntText(value=DEFAULT_PARAMS['profile_order']),
    tail_start=widgets.FloatText(value=DEFAULT_PARAMS['tail_start']),
    seed=widgets.IntText(value=0),
    show_empirical=widgets.Checkbox(value=True, description='Empirical SVD'),
    show_theory=widgets.Checkbox(value=True, description='Column-scaled theory'),
    show_tail=widgets.Checkbox(value=True, description='Tail asymptotic'),
    show_constant_check=widgets.Checkbox(value=True, description='Constant-profile check'),
)
def explore_column_scaled_singular_values(
    alpha,
    gamma,
    profile,
    constant_value,
    left_value,
    right_value,
    split,
    exponent,
    cutoff,
    entry_scale,
    normalization,
    n_rows,
    num_matrices,
    bins,
    s_max,
    num_points,
    imag_eps,
    quadrature_order,
    profile_order,
    tail_start,
    seed,
    show_empirical,
    show_theory,
    show_tail,
    show_constant_check,
):
    plot_column_scaled_comparison(
        alpha=alpha,
        gamma=gamma,
        profile=profile,
        constant_value=constant_value,
        left_value=left_value,
        right_value=right_value,
        split=split,
        exponent=exponent,
        cutoff=cutoff,
        entry_scale=entry_scale,
        normalization=normalization,
        n_rows=n_rows,
        num_matrices=num_matrices,
        bins=bins,
        s_max=s_max,
        num_points=num_points,
        imag_eps=imag_eps,
        quadrature_order=quadrature_order,
        profile_order=profile_order,
        tail_start=tail_start,
        seed=seed,
        show_empirical=show_empirical,
        show_theory=show_theory,
        show_tail=show_tail,
        show_constant_check=show_constant_check,
    )
